# Modified Swiss Dwellings Part 2: Multi-Unit Analysis and Visualization

This notebook continues the MSD analysis, focusing on:
- Multiple apartment units
- Unit-level analysis
- Export and visualization workflows

**Adapted from topologicpy MSD02 notebook**

The Modified Swiss Dwellings is a machine learning-ready floor plan dataset.
Dataset license: CC BY-SA 4.0

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

## Color and Type Definitions

In [ ]:
# Apartment unit colors
APARTMENT_COLORS = [
    '#FF0000', '#FF7F00', '#FFBF00', '#FFFF00',
    '#BFFF00', '#7FFF00', '#00FF00', '#00FF7F',
    '#00FFFF', '#007FFF', '#0000FF', '#7F00FF'
]

# Room type colors
ROOM_COLORS = {
    'Bedroom': '#1f77b4',
    'Livingroom': '#e6550d',
    'Kitchen': '#fd8d3c',
    'Dining': '#fdae6b',
    'Corridor': '#fdd0a2',
    'Stairs': '#72246c',
    'Storeroom': '#5254a3',
    'Bathroom': '#6b6ecf',
    'Balcony': '#2ca02c',
}

# Zone assignments
ZONE_INFO = {
    'Bedroom': ('Zone1', 'Private'),
    'Livingroom': ('Zone2', 'Living'),
    'Kitchen': ('Zone2', 'Living'),
    'Dining': ('Zone2', 'Living'),
    'Corridor': ('Zone2', 'Living'),
    'Stairs': ('Zone3', 'Service'),
    'Storeroom': ('Zone3', 'Service'),
    'Bathroom': ('Zone3', 'Service'),
    'Balcony': ('Zone4', 'Outdoor'),
}

print("Room types:", list(ROOM_COLORS.keys()))

## Create Multi-Unit Floor Plan

This simulates a building floor with multiple apartment units.

In [ ]:
# Define multiple apartment units
# Each unit has: unit_id, offset_x, offset_y, rooms list

def create_apartment_type_a(unit_id, offset_x, offset_y):
    """Type A: 2-bedroom apartment"""
    rooms = [
        ('Bedroom', 0, 4, 3, 3),
        ('Bedroom', 3, 4, 3, 3),
        ('Bathroom', 6, 4, 2, 3),
        ('Livingroom', 0, 0, 5, 4),
        ('Kitchen', 5, 0, 3, 4),
    ]
    return [(unit_id, rtype, x + offset_x, y + offset_y, w, h) for rtype, x, y, w, h in rooms]

def create_apartment_type_b(unit_id, offset_x, offset_y):
    """Type B: 1-bedroom apartment"""
    rooms = [
        ('Bedroom', 0, 4, 4, 3),
        ('Bathroom', 4, 4, 2, 3),
        ('Livingroom', 0, 0, 4, 4),
        ('Kitchen', 4, 0, 2, 4),
    ]
    return [(unit_id, rtype, x + offset_x, y + offset_y, w, h) for rtype, x, y, w, h in rooms]

# Create building floor with 4 units
all_rooms = []

# Unit 1: Type A (left side, bottom)
all_rooms.extend(create_apartment_type_a(1, 0, 0))

# Unit 2: Type B (right side, bottom)
all_rooms.extend(create_apartment_type_b(2, 10, 0))

# Unit 3: Type A (left side, top)
all_rooms.extend(create_apartment_type_a(3, 0, 9))

# Unit 4: Type B (right side, top)
all_rooms.extend(create_apartment_type_b(4, 10, 9))

# Add common corridor
corridor_rooms = [
    (0, 'Corridor', 8, 0, 2, 7),   # Vertical corridor (common)
    (0, 'Corridor', 8, 9, 2, 7),   # Vertical corridor (common, top)
    (0, 'Stairs', 8, 7, 2, 2),     # Central stairwell
]
all_rooms.extend(corridor_rooms)

print(f"Created {len(all_rooms)} rooms in {4} apartment units + common areas")

In [ ]:
# Process rooms into structured data
room_info = []

for i, (unit_id, rtype, x, y, w, h) in enumerate(all_rooms):
    zone, zone_name = ZONE_INFO.get(rtype, ('Unknown', 'Unknown'))
    
    info = {
        'id': i,
        'unit_id': unit_id,
        'type': rtype,
        'x': x,
        'y': y,
        'width': w,
        'height': h,
        'centroid': (x + w/2, y + h/2),
        'area': w * h,
        'room_color': ROOM_COLORS.get(rtype, '#999999'),
        'unit_color': APARTMENT_COLORS[unit_id % len(APARTMENT_COLORS)] if unit_id > 0 else '#aaaaaa',
        'zone': zone,
        'zone_name': zone_name
    }
    room_info.append(info)

# Summary by unit
print("Unit Summary:")
print("=" * 50)

for unit_id in range(5):  # 0 = common, 1-4 = apartments
    unit_rooms = [r for r in room_info if r['unit_id'] == unit_id]
    total_area = sum(r['area'] for r in unit_rooms)
    room_types = [r['type'] for r in unit_rooms]
    
    if unit_id == 0:
        label = "Common Areas"
    else:
        label = f"Unit {unit_id}"
    
    print(f"\n{label}:")
    print(f"  Rooms: {len(unit_rooms)}")
    print(f"  Area: {total_area:.1f} m^2")
    print(f"  Types: {', '.join(room_types)}")

## Define Room Adjacencies

For a multi-unit building, we need to define adjacencies within and between units.

In [ ]:
def rooms_are_adjacent(r1, r2, tolerance=0.1):
    """
    Check if two rooms share a wall (are adjacent).
    Uses simple axis-aligned bounding box overlap check.
    """
    # Get bounding boxes
    x1_min, y1_min = r1['x'], r1['y']
    x1_max, y1_max = r1['x'] + r1['width'], r1['y'] + r1['height']
    
    x2_min, y2_min = r2['x'], r2['y']
    x2_max, y2_max = r2['x'] + r2['width'], r2['y'] + r2['height']
    
    # Check if they share an edge
    # Horizontal adjacency (share vertical edge)
    if abs(x1_max - x2_min) < tolerance or abs(x2_max - x1_min) < tolerance:
        # Check y overlap
        if y1_max > y2_min + tolerance and y2_max > y1_min + tolerance:
            return True
    
    # Vertical adjacency (share horizontal edge)
    if abs(y1_max - y2_min) < tolerance or abs(y2_max - y1_min) < tolerance:
        # Check x overlap
        if x1_max > x2_min + tolerance and x2_max > x1_min + tolerance:
            return True
    
    return False

# Calculate all adjacencies
adjacencies = []
for i, r1 in enumerate(room_info):
    for j, r2 in enumerate(room_info):
        if i < j:  # Avoid duplicates
            if rooms_are_adjacent(r1, r2):
                adjacencies.append((i, j))

print(f"Found {len(adjacencies)} room adjacencies")

## Create Building Graph

In [ ]:
# Create vertices
vertices = []
for info in room_info:
    cx, cy = info['centroid']
    v = tf.Vertex.ByCoordinates(cx, cy, 0)
    vertices.append(v)

# Create edges from adjacencies
edges = []
for i, j in adjacencies:
    edge = tf.Edge.ByStartVertexEndVertex(vertices[i], vertices[j])
    edges.append(edge)

# Create graph
graph = tf.Graph.ByVerticesEdges(vertices, edges)

print(f"Building Graph:")
print(f"  Rooms (vertices): {graph.Order()}")
print(f"  Adjacencies (edges): {graph.Size()}")
print(f"  Density: {graph.Density():.3f}")
print(f"  Diameter: {graph.Diameter()} steps")
print(f"  Is Connected: {graph.MinimumDelta() > 0}")

## Visualization: By Room Type

In [ ]:
def visualize_building(room_info, graph, color_key='room_color', title='Building Floor Plan'):
    """Create visualization with specified coloring."""
    fig = go.Figure()
    
    # Draw rooms
    for info in room_info:
        x = info['x']
        y = info['y']
        w = info['width']
        h = info['height']
        
        rect_x = [x, x+w, x+w, x, x]
        rect_y = [y, y, y+h, y+h, y]
        
        color = info.get(color_key, '#999999')
        
        unit_label = f"Unit {info['unit_id']}" if info['unit_id'] > 0 else "Common"
        
        fig.add_trace(go.Scatter(
            x=rect_x, y=rect_y,
            fill='toself',
            fillcolor=color,
            line=dict(color='black', width=1),
            name=f"{info['type']} ({unit_label})",
            hoverinfo='name',
            opacity=0.8,
            showlegend=False
        ))
    
    # Draw graph edges
    graph_edges = graph.Edges()
    for edge in graph_edges:
        verts = edge.Vertices()
        if len(verts) == 2:
            p1 = verts[0].Coordinates()
            p2 = verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='black', width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices
    for info in room_info:
        cx, cy = info['centroid']
        fig.add_trace(go.Scatter(
            x=[cx], y=[cy],
            mode='markers',
            marker=dict(size=10, color='white', line=dict(color='black', width=1)),
            showlegend=False,
            hovertext=f"{info['type']} (Unit {info['unit_id']}): {info['area']:.0f} m^2",
            hoverinfo='text'
        ))
    
    fig.update_layout(
        title=title,
        xaxis=dict(title='X (m)', scaleanchor='y', scaleratio=1, range=[-1, 18]),
        yaxis=dict(title='Y (m)', range=[-1, 17]),
        width=900,
        height=700
    )
    
    return fig

fig_rooms = visualize_building(room_info, graph, 'room_color', 'Building by Room Type')
fig_rooms.show()

## Visualization: By Apartment Unit

In [ ]:
fig_units = visualize_building(room_info, graph, 'unit_color', 'Building by Apartment Unit')
fig_units.show()

## Per-Unit Graph Analysis

In [ ]:
# Create subgraph for each unit
print("Per-Unit Graph Analysis:")
print("=" * 60)

for unit_id in range(1, 5):  # Units 1-4
    # Get room indices for this unit
    unit_room_indices = [i for i, r in enumerate(room_info) if r['unit_id'] == unit_id]
    unit_rooms = [room_info[i] for i in unit_room_indices]
    
    # Create unit vertices
    unit_vertices = [vertices[i] for i in unit_room_indices]
    
    # Filter edges to only those within this unit
    unit_edges = []
    for i, j in adjacencies:
        if i in unit_room_indices and j in unit_room_indices:
            # Remap to local indices
            local_i = unit_room_indices.index(i)
            local_j = unit_room_indices.index(j)
            edge = tf.Edge.ByStartVertexEndVertex(unit_vertices[local_i], unit_vertices[local_j])
            unit_edges.append(edge)
    
    if len(unit_vertices) > 0 and len(unit_edges) > 0:
        unit_graph = tf.Graph.ByVerticesEdges(unit_vertices, unit_edges)
        
        print(f"\nUnit {unit_id}:")
        print(f"  Rooms: {unit_graph.Order()}")
        print(f"  Internal connections: {unit_graph.Size()}")
        print(f"  Density: {unit_graph.Density():.3f}")
        print(f"  Diameter: {unit_graph.Diameter()} steps")
        print(f"  Room types: {', '.join([r['type'] for r in unit_rooms])}")

## Cross-Unit Connectivity Analysis

In [ ]:
# Find edges that connect different units (through common areas)
cross_unit_edges = []

for i, j in adjacencies:
    unit_i = room_info[i]['unit_id']
    unit_j = room_info[j]['unit_id']
    
    if unit_i != unit_j:
        cross_unit_edges.append((i, j, unit_i, unit_j))

print("Cross-Unit Connections (through common areas):")
print("=" * 60)

for i, j, unit_i, unit_j in cross_unit_edges:
    room_i = room_info[i]
    room_j = room_info[j]
    
    unit_label_i = f"Unit {unit_i}" if unit_i > 0 else "Common"
    unit_label_j = f"Unit {unit_j}" if unit_j > 0 else "Common"
    
    print(f"  {room_i['type']} ({unit_label_i}) <-> {room_j['type']} ({unit_label_j})")

## Inter-Unit Path Finding

In [ ]:
# Find paths between rooms in different units

def find_bedroom_in_unit(unit_id):
    """Find a bedroom room index in the given unit."""
    for i, r in enumerate(room_info):
        if r['unit_id'] == unit_id and r['type'] == 'Bedroom':
            return i
    return None

# Path from Unit 1 bedroom to Unit 4 bedroom
start_idx = find_bedroom_in_unit(1)
end_idx = find_bedroom_in_unit(4)

if start_idx is not None and end_idx is not None:
    start_v = vertices[start_idx]
    end_v = vertices[end_idx]
    
    distance = graph.Distance(start_v, end_v)
    path = graph.Path(start_v, end_v)
    
    print(f"Path from Unit 1 Bedroom to Unit 4 Bedroom:")
    print("=" * 50)
    print(f"  Distance: {distance} rooms")
    
    if path:
        path_verts = path.Vertices()
        print(f"  Route:")
        for pv in path_verts:
            coords = pv.Coordinates()
            # Find matching room
            for r in room_info:
                cx, cy = r['centroid']
                if abs(coords[0] - cx) < 0.1 and abs(coords[1] - cy) < 0.1:
                    unit_label = f"Unit {r['unit_id']}" if r['unit_id'] > 0 else "Common"
                    print(f"    -> {r['type']} ({unit_label})")
                    break

## Building Statistics

In [ ]:
# Calculate various building statistics
total_area = sum(r['area'] for r in room_info)
unit_areas = {}
for r in room_info:
    uid = r['unit_id']
    if uid not in unit_areas:
        unit_areas[uid] = 0
    unit_areas[uid] += r['area']

type_areas = {}
for r in room_info:
    rtype = r['type']
    if rtype not in type_areas:
        type_areas[rtype] = 0
    type_areas[rtype] += r['area']

print("Building Statistics:")
print("=" * 50)
print(f"\nTotal Rooms: {len(room_info)}")
print(f"Total Area: {total_area:.1f} m^2")
print(f"Total Adjacencies: {len(adjacencies)}")
print(f"Cross-Unit Connections: {len(cross_unit_edges)}")

print(f"\nArea by Unit:")
for uid in sorted(unit_areas.keys()):
    label = f"Unit {uid}" if uid > 0 else "Common"
    pct = 100 * unit_areas[uid] / total_area
    print(f"  {label}: {unit_areas[uid]:.1f} m^2 ({pct:.1f}%)")

print(f"\nArea by Room Type:")
for rtype in sorted(type_areas.keys(), key=lambda x: -type_areas[x]):
    pct = 100 * type_areas[rtype] / total_area
    print(f"  {rtype}: {type_areas[rtype]:.1f} m^2 ({pct:.1f}%)")

In [ ]:
# Create statistics visualization
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['Area by Room Type', 'Area by Unit']
)

# Room type pie
fig.add_trace(
    go.Pie(
        labels=list(type_areas.keys()),
        values=list(type_areas.values()),
        marker_colors=[ROOM_COLORS.get(t, '#999999') for t in type_areas.keys()],
        hole=0.3
    ),
    row=1, col=1
)

# Unit pie
unit_labels = [f"Unit {u}" if u > 0 else "Common" for u in unit_areas.keys()]
unit_colors = [APARTMENT_COLORS[u % len(APARTMENT_COLORS)] if u > 0 else '#aaaaaa' for u in unit_areas.keys()]
fig.add_trace(
    go.Pie(
        labels=unit_labels,
        values=list(unit_areas.values()),
        marker_colors=unit_colors,
        hole=0.3
    ),
    row=1, col=2
)

fig.update_layout(
    title='Building Area Distribution',
    width=900,
    height=400
)

fig.show()

## Summary

This notebook demonstrated:

1. **Multi-Unit Floor Plans**: Creating building floors with multiple apartment units
2. **Automatic Adjacency Detection**: Computing room adjacencies from geometry
3. **Building Graph**: Creating a unified connectivity graph
4. **Per-Unit Analysis**: Analyzing subgraphs for individual units
5. **Cross-Unit Connectivity**: Understanding how units connect via common areas
6. **Path Finding**: Routing between rooms in different units
7. **Statistics**: Computing area distributions

### Not Yet Implemented in topologic_fast:
- `Topology.SetDictionary()` / `Dictionary` class - Storing attributes on topologies
- `Topology.ByJSONPath()` - Import from JSON files
- `Graph.ExportToJSON()` - Export graph to JSON
- `Plotly.ExportToImage()` - Export figures to PNG

### Applications:
- Building floor plan analysis
- Multi-family housing design
- Wayfinding in buildings
- Space syntax research
- ML training data preparation